In [ ]:
# a & e data are read in from .csv files

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from ReadParticle import read_particle, read_particle_frames
from datetime import datetime, timedelta

# Read particle and collision data
#Np_seq, time, N_colDidy, N_colDimor, N_escape, r_dust, particle, data_c, data_p = read_particle('particles.txt', 'collide.txt')
Np_seq, time, r_dust, data_p = read_particle_frames('/home/linfel/rebound_exp/test_ejecta_180_datahigh/particles.txt')

In [ ]:
"""Top view Parallel rendering and make video"""

import os
from multiprocessing import Pool
import numpy as np
from frame_renderer import init_worker, render_frame_topview

# Ensure output directory exists for saving frames
output_dir = "/home/linfel/rebound_exp/test_ejecta_180_datahigh_speedreduced/postprocess/topview_frames"
#output_dir = "/home/linfel/rebound_exp/test_ejecta_160/postprocess/topview_frames"

if os.path.exists(output_dir):
    import shutil
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

# Parameters
#num_frames = len(time)
num_frames = 150
fps = 24
zmout_freq = 1000      # zoom out every # of frames
output_name = "/home/linfel/rebound_exp/test_ejecta_180_datahigh_speedreduced/postprocess/topview_movie_closer.mp4"

# set the plot axis limit for each frame
axis_lim_list = [30e3] * num_frames
# for i in range(num_frames): # Must be executed sequentially 
#     if i%zmout_freq==0:
#         data_i = data_p[i]
#         dust_distance = (data_i[4:, 1]**2 + data_i[4:, 2]**2 + data_i[4:, 3]**2)**0.5
#         dis_max = dust_distance.max()
#         axis_lim_list.append(dis_max)
#         cache1 = dis_max
#     else:
#         axis_lim_list.append(cache1)
#axis_lim_list = np.array(axis_lim_list)

if __name__ == "__main__": 
    # Render all frames in parallel
    with Pool(initializer=init_worker, initargs=(data_p, time, axis_lim_list, output_dir)) as pool:
        pool.map(render_frame_topview, range(num_frames))
    print("All frames rendered. Combining into video...")

    # Combine frames into an MP4 using ffmpeg
    os.system(f"ffmpeg -y -r {fps} -i {output_dir}/topview_frame_%04d.png -vf 'pad=ceil(iw/2)*2:ceil(ih/2)*2' -vcodec libx264 -crf 18 -pix_fmt yuv420p {output_name}")

    print(f"Animation saved as {output_name}",flush=True)

In [ ]:
"""HST view parallel rendering and make video"""

import os
import numpy as np
from multiprocessing import Pool
from frame_renderer import init_worker, render_frame_HSTview

# Ensure output directory exists for saving frames
output_dir = "HST_frames"
if os.path.exists(output_dir):
    import shutil
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

# Parameters
num_frames = len(time)
fps = 30
zmout_freq = 1000    # zoom out every # of frames
output_name = "HST_view.mp4"

# set the plot axis limit for each frame
axis_lim_list = []
for i in range(num_frames): # Must be executed sequentially 
    if i%zmout_freq==0:
        data_i = data_p[i]
        dust_distance = (data_i[4:, 1]**2 + data_i[4:, 2]**2 + data_i[4:, 3]**2)**0.5
        dis_max = dust_distance.max()
        axis_lim_list.append(dis_max)
        cache1 = dis_max
    else:
        axis_lim_list.append(cache1)

if __name__ == "__main__": 
    # Render all frames in parallel
    with Pool(initializer=init_worker, initargs=(data_p, time, axis_lim_list, output_dir)) as pool:
        pool.map(render_frame_HSTview, range(num_frames))
    print("All frames rendered. Combining into video...")

    # Combine frames into an MP4 using ffmpeg
    os.system(f"ffmpeg -y -r {fps} -i {output_dir}/frame_%04d.png -vf 'pad=ceil(iw/2)*2:ceil(ih/2)*2' -vcodec libx264 -crf 18 -pix_fmt yuv420p {output_name}")

    print(f"Animation of HSTview saved as {output_name}",flush=True)